# mysql-connector-python
- Python 프로그램에서 MySQL 데이터베이스에 접속하고 SQL을 실행할 수 있게 해주는 라이브러리

In [ ]:
%pip install mysql-connector-python

## Connection과 Cursor
- Connection : DB 서버에 연결하는 객체
- Cursor : 쿼리를 수행하고 결과를 가지고 있는 객체
- 두 객체 모두 사용 후 반납 필수!!!

In [ ]:
import mysql.connector

# DB 접속 정보를 전달하여 연결 객체 생성 후 반환
conn = mysql.connector.connect(
    host='localhost',
    port=3306,
    user='ohgiraffers',
    password='ohgiraffers',
    database='menudb'
)

# 연결 객체를 통해 커서 객체 생성
cursor = conn.cursor()

sql = """
select
    *
from
    tbl_menu
"""
cursor.execute(sql)         # 전달한 sql 구문 실행
rows = cursor.fetchall()    # 결과 집합(result set) 반환
for row in rows:
    print(row)              # 한 행은 튜플 형태로 반환
    
# 연결 종료 (리소스 반환)
cursor.close()
conn.close()    


## Menu 클래스 정의 후 객체에 데이터 담기

In [ ]:
# 데이터를 저장하는 목적의 클래스를 쉽게 만들도록 도와주는 기능으로
# 일반 클래스에선 __init__, __repr__ 등을 직접 작성해야 하지만 자동으로 생성해준다.
from dataclasses import dataclass

# typing 모듈 타입 힌트를 작성할 때 사용하는 도구 제공
from typing import Optional

@dataclass
class Menu:
    
    menu_code: int
    menu_name: str
    menu_price: int
    # int 또는 None이 들어올 수 있다는 의미
    category_code: Optional[int]
    orderable_status: str

In [27]:
# Menu 인스턴스에 조회 된 데이터 담기
db_config = {
    "host": "localhost",
    "port": 3306,
    "user": "ohgiraffers",
    "password": "ohgiraffers",
    "database": "menudb"
}

# 인스턴스를 담을 리스트
menus = []
with mysql.connector.connect(**db_config) as conn:
    with conn.cursor() as cursor:
        cursor.execute("select * from tbl_menu")
        rows = cursor.fetchall()
        for row in rows:
            menus.append(Menu(*row))
            
# 메뉴 리스트 출력
menus

[Menu(menu_code=1, menu_name='테스트', menu_price=4500, category_code=10, orderable_status='Y'),
 Menu(menu_code=2, menu_name='우럭쥬스', menu_price=6000, category_code=9, orderable_status='N'),
 Menu(menu_code=3, menu_name='생갈치쉐이크', menu_price=6000, category_code=10, orderable_status='Y'),
 Menu(menu_code=4, menu_name='갈릭미역파르페', menu_price=7000, category_code=10, orderable_status='Y'),
 Menu(menu_code=5, menu_name='앙버터김치찜', menu_price=13000, category_code=4, orderable_status='N'),
 Menu(menu_code=6, menu_name='생마늘샐러드', menu_price=12000, category_code=4, orderable_status='Y'),
 Menu(menu_code=7, menu_name='민트미역국', menu_price=15000, category_code=4, orderable_status='Y'),
 Menu(menu_code=8, menu_name='한우딸기국밥', menu_price=20000, category_code=4, orderable_status='Y'),
 Menu(menu_code=9, menu_name='홍어마카롱', menu_price=9000, category_code=12, orderable_status='Y'),
 Menu(menu_code=10, menu_name='코다리마늘빵', menu_price=7000, category_code=12, orderable_status='N'),
 Menu(menu_code=11, menu_name='정어리빙수

## 한 건 조회 (메뉴 코드 기준)

In [ ]:
try:
    with mysql.connector.connect(**db_config) as conn:
        with conn.cursor() as cursor:
            sql = """
                select
                    *
                from
                    tbl_menu
                where
                    menu_code = %s
            """
            cursor.execute(sql, (5,))        # sql, params
            row = cursor.fetchone()
            print(Menu(*row))
except mysql.connector.Error as e:
    print('DB 오류', e)

## 여러 건 조회 (메뉴 이름 키워드 포함)

In [ ]:
try:
    with mysql.connector.connect(**db_config) as conn:
        with conn.cursor() as cursor:
            sql = """
                select
                    *
                from
                    tbl_menu
                where
                    menu_name like concat('%', %s, '%')
            """
            cursor.execute(sql, ('마늘',))        # sql, params
            menus = [Menu(*row) for row in cursor.fetchall()]
            print(menus)
except mysql.connector.Error as e:
    print('DB 오류', e)

In [ ]:
menu = Menu(None, '요거트액젓', 20_000, 8, 'Y')
try:
    with mysql.connector.connect(**db_config) as conn:
        with conn.cursor() as cursor:
            sql = """
                insert into tbl_menu (menu_name, menu_price, category_code, orderable_status)
                values(%s, %s,%s, %s)
            """
            cursor.execute(
                sql, 
                (menu.menu_name, menu.menu_price, menu.category_code, menu.orderable_status)
            )        # sql, params
            print(cursor.rowcount, '개 행이 삽입 되었습니다.')  # rowcount : DML을 통해 변경 된 행의 개수
            conn.commit()   # autocommit이 아니므로 명시적 commit 호출
except mysql.connector.Error as e:
    print('DB 오류', e)

## 레코드 수정
- 삽입 했던 행의 메뉴 코드를 이용해서 메뉴명 수정하기

In [ ]:
menu_code = 36
new_menu_name = '울트라요거트액젓'

try:
    with mysql.connector.connect(**db_config) as conn:
        with conn.cursor() as cursor:
            sql = """
                update 
                    tbl_menu
                set 
                    menu_name = %s
                where 
                    menu_code = %s
            """
            cursor.execute(
                sql, 
                (new_menu_name, menu_code)
            )
            print(cursor.rowcount, '개 행이 수정 되었습니다.')
            conn.commit()
except mysql.connector.Error as e:
    print('DB 오류', e)

## 레코드 삭제
- 삽입 했던 행의 메뉴 코드를 이용해서 행 삭제

In [ ]:
menu_code = 36

try:
    with mysql.connector.connect(**db_config) as conn:
        with conn.cursor() as cursor:
            sql = """
                delete from 
                    tbl_menu
                where 
                    menu_code = %s
            """
            cursor.execute(
                sql, 
                (menu_code,)
            )
            print(cursor.rowcount, '개 행이 삭제 되었습니다.')
            conn.commit()
except mysql.connector.Error as e:
    print('DB 오류', e)

1 개 행이 삭제 되었습니다.
